# Prepare MAMS profiles with MESA

This notebook turns the existing `tutorial/inlist_project` into a safe mass-grid workflow. Each mass gets an isolated directory and unique MESA-profile and GYRE filenames, so reruns cannot overwrite models for other masses.

The notebook only **prepares** runs by default. Expensive MESA evolution or Slurm submission happens only when you explicitly enable it in the parameter cell.

## Folder layout and command-line equivalent

```text
scripts/mams_profiles/
  pipeline/mams_pipeline.py       # reusable implementation and CLI
  notebook/mams_profiles.ipynb    # this walkthrough
results/mams_profiles/
  manifest.json
  MAMS1Msun/                      # isolated MESA work directory
  MAMS1p5Msun/
```

From the repository root, setup can also be done without Jupyter:

```bash
python3 scripts/mams_profiles/pipeline/mams_pipeline.py --masses 1.0 1.5 2.0
```

In [ ]:
# Locate the project whether Jupyter starts here or at the repository root.
import json
import os
import sys
from pathlib import Path

def locate_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'tutorial' / 'inlist_project').is_file():
            return candidate
    raise FileNotFoundError('Could not locate tutorial/inlist_project')

PROJECT_ROOT = locate_project_root()
PIPELINE_DIR = PROJECT_ROOT / 'scripts' / 'mams_profiles' / 'pipeline'
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from mams_pipeline import (
    build_configs, inspect_runs, model_label, prepare_grid,
    render_inlist_project, run_local, submit_slurm,
)

print(f'Project root: {PROJECT_ROOT}')

## 1. Edit the grid and physics controls

These defaults reproduce the current project: pre-main-sequence evolution continues until central hydrogen falls below 0.35, with $Y=0.2703$ and $Z=0.0142$. The remaining physics settings—including the network, MLT, overshooting, and GYRE options—are inherited from `tutorial/inlist_project`.

`OVERWRITE=True` deletes an existing generated run directory before rebuilding it, including any MESA results inside it. Leave it false unless replacement is intentional.

In [ ]:
# Main user-editable parameters
MASSES_MSUN = [1.0, 1.5, 2.0]
INITIAL_Y = 0.2703
INITIAL_Z = 0.0142
CENTRAL_H1_LIMIT = 0.35

# Slurm settings used only if SUBMIT_SLURM is enabled.
PARTITION = 'typhon'
WALLTIME = '7-00:00:00'
CPUS_PER_TASK = 1

# An environment override makes validation and alternate output locations easy.
OUTPUT_ROOT = Path(os.environ.get(
    'MAMS_OUTPUT_ROOT', PROJECT_ROOT / 'results' / 'mams_profiles'
)).resolve()
TEMPLATE_DIR = PROJECT_ROOT / 'tutorial'
OVERWRITE = False
PREPARE_RUNS = True
RUN_LOCAL = False       # Runs models sequentially and waits; potentially slow.
SUBMIT_SLURM = False    # Submits all prepared models and returns job IDs.

if RUN_LOCAL and SUBMIT_SLURM:
    raise ValueError('Choose local execution or Slurm submission, not both.')

## 2. Build and inspect the configurations

This stage validates masses, composition, termination threshold, and output labels before touching the filesystem. Decimal points become `p` in filenames, matching the existing convention.

In [ ]:
configs = build_configs(
    MASSES_MSUN,
    initial_y=INITIAL_Y,
    initial_z=INITIAL_Z,
    central_h1_limit=CENTRAL_H1_LIMIT,
    partition=PARTITION,
    walltime=WALLTIME,
    cpus_per_task=CPUS_PER_TASK,
)

for config in configs:
    print(f'{config.mass_msun:8g} M_sun  ->  {model_label(config.mass_msun)}')

## 3. Preview one rendered inlist

The implementation patches only the output names, mass, composition, opacity metallicity, and central-hydrogen stopping condition. This preview is an in-memory smoke test; it does not create or launch a model.

In [ ]:
template_text = (TEMPLATE_DIR / 'inlist_project').read_text()
preview = render_inlist_project(template_text, configs[0])
keys_to_show = (
    'filename_for_profile_when_terminate', 'save_pulse_data_filename',
    'initial_mass', 'initial_y', 'initial_z', 'Zbase',
    'xa_central_lower_limit(1)',
)
for line in preview.splitlines():
    if any(key in line for key in keys_to_show):
        print(line)

assert f"'{model_label(configs[0].mass_msun)}.profile'" in preview
assert f"'{model_label(configs[0].mass_msun)}.mesa'" in preview
print('In-memory smoke test passed.')

## 4. Prepare isolated run directories

Each directory receives its own inlists, metadata, `run.sh`, and `submit.slurm`. The compiled `tutorial/star` executable is linked rather than rebuilt for every mass. Re-executing this cell with identical settings is idempotent.

In [ ]:
runs = []
if PREPARE_RUNS:
    runs = prepare_grid(
        configs, TEMPLATE_DIR, OUTPUT_ROOT, overwrite=OVERWRITE
    )
    print(f'Prepared {len(runs)} run(s) under {OUTPUT_ROOT}')
else:
    print('PREPARE_RUNS is false; no directories were written.')

inspect_runs(runs)

## 5. Optionally run MESA or submit to Slurm

Local runs are sequential. Slurm submissions are independent because every job has a separate working directory. Change the two launch flags in the parameter cell and rerun from that cell onward when ready.

In [ ]:
job_ids = {}
if RUN_LOCAL:
    for run in runs:
        print(f'Running {run.label} ...', flush=True)
        run_local(run)
elif SUBMIT_SLURM:
    for run in runs:
        job_ids[run.label] = submit_slurm(run)
    print(json.dumps(job_ids, indent=2))
else:
    print('Setup only: no MESA processes or Slurm jobs were started.')

## 6. Inspect generated artifacts

Immediately after setup, metadata should exist while the two scientific outputs remain absent. After a successful MESA termination, both `mesa_profile` and `gyre_model` become true. The GYRE header reports zone count, mass, radius, luminosity, and format version.

In [ ]:
status = inspect_runs(runs)
print(json.dumps(status, indent=2))

if runs:
    metadata = json.loads(runs[0].metadata_path.read_text())
    print('\nFirst run metadata:')
    print(json.dumps(metadata, indent=2))

    if runs[0].gyre_path.is_file():
        fields = runs[0].gyre_path.read_text().splitlines()[0].split()
        names = ['zones', 'mass_g', 'radius_cm', 'luminosity_erg_s', 'version']
        print('\nGYRE header:', dict(zip(names, fields)))

## Interpretation

A prepared run is only a configured work directory. A completed run should contain both `MAMS<mass>Msun.profile` (the regular MESA structure profile) and `MAMS<mass>Msun.mesa` (the GYRE pulsation model). MESA may step slightly below the requested $X_c(\mathrm{H}^1)=0.35$ threshold before terminating; that matches the behavior of the existing models.